# HBI&nbsp;01 — Synthetic end-to-end closure

**What this notebook does (in one breath):** we *invent* a known column-density
distribution $f(N)$, *forward-simulate* a DLA detection catalog from it (the
same physical effects the GP pipeline would impose — incompleteness and an
Eddington measurement smear), then run the **real production MAP deconvolution**
on that synthetic catalog and check we recover the $f(N)$ we put in.

```
   inject  f_b_true (known power law)
      │
      ▼  forward-simulate
   catalog of "measured" detections  N̂_i   (completeness-thinned + Eddington-smeared)
      │
      ▼  REAL production estimator  (cddf_catalog_hbi)
   MAP-deconvolved  f_rec
      │
      ▼  self-check
   dN/dX(≥20.3)_rec  ≈  dN/dX(≥20.3)_true   (within 8%)
```

**No GP inference. No scratch caches. No GPFS.** Everything runs in a couple of
seconds on a laptop. This is the cleanest possible setting to *see the math
work*: because the truth is something we chose, every intermediate quantity has
a known right answer, and the closure assertion at the end is checked against
the **injected truth**, not against a self-consistency recovery.

This notebook is a faithful, narrated adaptation of the production test
`tests/test_cddf_catalog_hbi.py::test_v2_synthetic_closure_recovers_injected_fb`
(the `_FakeXcalc` cosmology stub and `_make_cfg` config wrapper are copied
verbatim). Every printed number is computed by the code in the cell — nothing is
typed in.

> Companion notebooks: **NB2** swaps the synthetic forward kernel for the real
> 2LPT-0 forward kernel; **NB3** opens up the likelihood, the MAP solve, and the
> Monte-Carlo band.

In [1]:
# --- ENV: pin BLAS to 1 thread BEFORE numpy is imported ----------------------
# The MAP solve is an L-BFGS-B over a sparse normal-equations system; its result
# is sensitive to BLAS thread scheduling, so we pin every threadpool to 1 for
# bit-for-bit reproducibility (this matches the production test harness).
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = "1"

# --- resolve repo root so the import works no matter the launch dir ----------
import sys
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in (_here, *_here.parents)
              if (p / "CDDF_analysis" / "hbi" / "cddf_catalog_hbi.py").exists()),
             None)
assert _root is not None, "could not locate repo root (CDDF_analysis/hbi/...)"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt

# THE canonical production estimator module — the same code that runs at scale.
from CDDF_analysis.hbi import cddf_catalog_hbi as H

print("repo root :", _root)
print("estimator :", H.__file__)

repo root : /tmp/claude-114399728/-home-mfho-desi-gpy-dla-detection/61bff2ab-596e-490e-9db3-2dbe8f55d4d9/scratchpad/hbi_tutorial_wt
estimator : /tmp/claude-114399728/-home-mfho-desi-gpy-dla-detection/61bff2ab-596e-490e-9db3-2dbe8f55d4d9/scratchpad/hbi_tutorial_wt/CDDF_analysis/hbi/cddf_catalog_hbi.py


## A tiny cosmology stub

The estimator needs the absorption-distance element $dX/dz=(1+z)^2/E(z)$ to turn
a redshift window into a path length $\Delta X$ (the denominator of $dN/dX$).
For this synthetic test we only need *relative* path lengths, so we use a
minimal stand-in (`_FakeXcalc`) instead of the full `AbsorptionDistance` class.
It is copied **verbatim** from the test (lines 261–287).

In [2]:
class _FakeXcalc:
    """Minimal AbsorptionDistance stand-in with the dX/dz = (1+z)²/E(z) integral
    needed by build_M_b (deltaX) and the cosmology E(z)."""
    def __init__(self, Omega_m=0.279):
        self.Omega_m = Omega_m
        self._cache = {}

    def _E(self, z):
        return np.sqrt(self.Omega_m * (1 + z) ** 3 + (1 - self.Omega_m))

    def X(self, z):
        # numeric integral of (1+z)²/E from 0 to z (coarse; only relative used)
        key = round(float(z), 6)
        if key not in self._cache:
            zg = np.linspace(0, key, 2000)
            _trap = getattr(np, "trapezoid", getattr(np, "trapz", None))
            self._cache[key] = float(_trap((1 + zg) ** 2 / self._E(zg), zg))
        return self._cache[key]

    def deltaX(self, z1, z2):
        # vectorized + memoized on unique (z1,z2) pairs (identical windows -> fast)
        z1 = np.atleast_1d(np.asarray(z1, float))
        z2 = np.atleast_1d(np.asarray(z2, float))
        out = np.empty(len(z1))
        for i in range(len(z1)):
            out[i] = self.X(z2[i]) - self.X(z1[i])
        return out

## Configuration and the fine $(\log N, z)$ grid

`HBIConfig` is the production config dataclass. We build a minimal instance with
the **exact kwargs the closure test uses** (dummy paths — we never load a real
catalog here). The notable choices, all chosen to make the *closure* clean:

| kwarg | value | why |
|---|---|---|
| `logN_lo`, `logN_hi` | 20.0, 21.5 | the DLA tier we fit |
| `drop_top_bin_above` | 21.4 | drop the open top bin (so the injected truth, which tops at 21.4, never leaks above the grid) |
| `v2_logN_fit_floor` | 20.0 | the forward kernel is trustworthy at/above this |
| `occupancy_floor` | 1 | keep every populated $N$-bin active |
| `v2_z_fit_lo/hi/step` | 2.4, 2.6, 0.2 | a **single** fine $z$-bin spanning [2.4, 2.6] — the synthetic truth puts all absorbers in one $z$-bin, so extra empty $z$-columns would just dilute the path length |
| `zbins` | (2.4, 2.6) | matching coarse $z$-binning |
| `report_logN_limits` | (20.0, 20.3) | reporting integration limits |
| `v2_lambda_grid` | (1e-3,) | a single smoothing $\lambda$ (no L-curve to scan) |
| `v2_n_restart` | 3 | a few L-BFGS-B multi-starts |

`build_fine_grid` returns, per fine $N$-bin: lower/upper $\log N$ edges, the
**linear** bin-center column density $N_b$, and the **linear** bin width
$\Delta N_b$ (note: $f(N)$ is a density in $N$, not in $\log N$, so the linear
$\Delta N$ matters).

In [3]:
def make_cfg(**kw):
    '''Minimal HBIConfig with dummy paths (never loaded here) — same wrapper the
    test uses (_make_cfg). The kwargs below are the closure-test defaults.'''
    defaults = dict(
        catalog_dir="/dev/null", truth_path="/dev/null",
        bal_cat_path="/dev/null", molly_tsv="/dev/null", out_dir="/tmp",
        logN_lo=17.2, logN_hi=22.5, dlogN=0.1, drop_top_bin_above=22.4,
        zbins=(2.0, 2.5, 3.0, 3.5), report_logN_limits=(20.0, 20.3),
        H0=70.0, Omega_m=0.279,
    )
    defaults.update(kw)
    return H.HBIConfig(**defaults)


# the EXACT kwargs the closure test passes
cfg = make_cfg(
    logN_lo=20.0, logN_hi=21.5, drop_top_bin_above=21.4,
    v2_logN_fit_floor=20.0, occupancy_floor=1,
    v2_z_fit_lo=2.4, v2_z_fit_hi=2.6, v2_z_fit_step=0.2,
    zbins=(2.4, 2.6), report_logN_limits=(20.0, 20.3),
    v2_lambda_grid=(1e-3,), v2_n_restart=3,
)

logN_lo, logN_hi, N_b, dN_b = H.build_fine_grid(cfg)
n_nbins = len(logN_lo)
Xcalc = _FakeXcalc(cfg.Omega_m)

print(f"fine N-bins         : {n_nbins}")
print(f"logN edges          : [{logN_lo[0]:.2f} .. {logN_hi[-1]:.2f}]")
print(f"resp_kind (default) : {cfg.resp_kind!r}  "
      f"(legacy kappa object; see honesty cell)")
print(f"single fine z-bin   : [{cfg.v2_z_fit_lo}, {cfg.v2_z_fit_hi}]")

fine N-bins         : 14
logN edges          : [20.00 .. 21.40]
resp_kind (default) : 'kappa'  (legacy kappa object; see honesty cell)
single fine z-bin   : [2.4, 2.6]


## The estimand: a marked-Poisson likelihood

The catalog is modelled as an inhomogeneous (marked) Poisson process over the
detection plane. Writing $f_b$ for the binned CDDF heights we want to infer, the
log-likelihood is

$$
\log L
= -\bigl(\mu_{\rm det} + \mu_{\rm FP}\bigr)
\;+\; \sum_i w_i \,\log\!\bigl(\lambda_{{\rm real},i} + \lambda_{{\rm FP},i}\bigr).
$$

The two pieces:

- **Per-detection rate** $\lambda_{{\rm real},i} = (A\,f)_i$. The matrix $A$ is
  the **forward-response / deconvolution operator**: row $i$ spreads each true
  $N$-bin's contribution onto detection $i$ through the measurement kernel
  $N(\hat N_i \mid N,\sigma_i^2)$ and the cosmological $dX/dz$ weight. So
  $\lambda_{{\rm real},i}$ is the expected rate of *real* detections at $i$ given
  the underlying $f$.

- **Expected total real count** $\mu_{\rm det} = M\cdot f$. The vector $M$ is the
  **path-length normalizer**: $M_b = \Delta N_b \sum_s \Delta X_{s}$ (summed over
  sightlines), i.e. how much survey path length each $N$-bin is searched over.

- **Completeness** $C(N,{\rm SNR})$ enters multiplicatively: it *scales up* $A$
  (we divide by $C$ to undo the thinning — $\times 1/C$ on the per-detection
  response) and *scales down* $M$ (only a fraction $C$ of the path length yields
  a detection — $\times C$ on the normalizer). In code this is `_apply_C_to_A`
  and `_apply_C_to_M`.

- **False positives** ($\mu_{\rm FP}$, $\lambda_{{\rm FP},i}$) are a known
  background; in *this* controlled test there are none, so both are zero.

Maximizing $\log L$ (with a small $D_2$ smoothing penalty $\lambda_{\rm smooth}\,
\lVert D_2 f\rVert^2$ for stability) over $f \ge 0$ is the **MAP deconvolution**.
That solve is `H._solve_one_lambda`. Everything below assembles $A$, $M$, the
penalty, and runs that exact solver.

## A known completeness $C(N,{\rm SNR})$ and the injected truth $f_b^{\rm true}$

**Completeness** (`MollyMatrix`): two SNR cells split at 4, one $N$-cell over
$[20.0, 21.5)$. We make $C$ rise with SNR (0.80 for low-SNR sightlines, 0.95 for
high-SNR) and **flat in $N$** within the DLA tier — deliberately no sharp $C$
step at the 20.3 reporting boundary (a step there would couple the kernel width
to the boundary and create a hard Eddington amplification that a *controlled*
closure test should not have to fight; that boundary case is exercised on the
real catalog). Purity is 1 (no false positives).

**Injected truth**: a power law $f_b^{\rm true}=A\,N_b^{\beta}$ with
$\beta=-1.8$, normalized so $f(10^{20.3})=10^{-22}$. Because the fit grid tops at
21.4 = the truth top, no true system smears in from above the grid.

In [4]:
# known completeness matrix (Molly-style): rows = SNR cells, cols = N cells
snr_edges = np.array([0, 4, np.inf])
nhi_edges = np.array([20.0, 21.5])
C_mat = np.array([[0.8],    # SNR in [0,4)   -> 80% complete
                  [0.95]])  # SNR in [4,inf) -> 95% complete
mm = H.MollyMatrix(snr_edges=snr_edges, nhi_edges=nhi_edges,
                   purity=np.ones_like(C_mat), completeness=C_mat)

# injected power-law CDDF heights, f(N) = A N^beta
beta = -1.8
A = 1.0e-22 / (10.0 ** 20.3) ** beta     # f(10^20.3) = 1e-22 by construction
f_b_true = A * N_b ** beta

print(f"beta                : {beta}")
print(f"f(10^20.3)          : {A * (10.0**20.3)**beta:.3e}  (=1e-22 by design)")
print(f"completeness (SNR<4): {C_mat[0,0]:.2f}   (SNR>=4): {C_mat[1,0]:.2f}")

beta                : -1.8
f(10^20.3)          : 1.000e-22  (=1e-22 by design)
completeness (SNR<4): 0.80   (SNR>=4): 0.95


## Forward-simulate the "measured" catalog

This is the generative step — exactly what the GP pipeline does to nature, here
done explicitly so we know the truth. For each SNR class and each true $N$-bin:

1. Draw the **true** number of absorbers as Poisson with mean
   $\mu = f_b^{\rm true}\,\Delta N_b\,\Delta X_c$ (path length $\Delta X_c$ = #
   sightlines in this SNR class $\times$ the per-sightline $\Delta X$).
2. Give each a true $\log N$ uniform within its bin.
3. **Thin by completeness**: keep each with probability $C(N_{\rm true},{\rm SNR})$.
4. **Eddington-smear** the survivors: the *measured* $\hat N = N_{\rm true} +
   \mathcal{N}(0,\sigma^2)$ with $\sigma=0.02$ dex — a sub-bin, *near-delta*
   kernel. This is the σ→0 anchor: the deconvolution should reduce to a plain
   $1/V_{\rm max}$ count, so this isolates the forward-model normalization
   ($A$/$M$) and the solve mechanics from genuine ill-conditioned deconvolution.

The result is `cat_op`, the dict of "measured" detections that the estimator
consumes — the stand-in for what the GP would hand us. We assert we got
$> 2000$ detections so the closure has decent statistics.

In [5]:
rng = np.random.default_rng(7)   # match the test exactly

# many sightlines so the searched path length (M_b) is large; window [2.4,2.6]
n_sl = 200000
qso_zlo = np.full(n_sl, 2.4)
qso_zhi = np.full(n_sl, 2.6)
qso_snr = rng.choice([2.5, 10.0], size=n_sl, p=[0.5, 0.5])
z_edges_fine = H._fine_z_grid(cfg)

dX_sl = float(Xcalc.deltaX(2.4, 2.6)[0])     # per-sightline path length
sigma_kernel = 0.02                          # sub-bin Eddington smear (near-delta)

# forward-simulate detections, SNR class by SNR class, N-bin by N-bin
obs_xhat = []; obs_snr = []; obs_zhat = []
C_interp = H.make_C_interpolator(mm)
for c, snr_cell in enumerate([2.5, 10.0]):
    n_sl_c = int((qso_snr == snr_cell).sum())
    dX_c = n_sl_c * dX_sl
    for b in range(n_nbins):
        mean_true = f_b_true[b] * dN_b[b] * dX_c     # expected TRUE count in this bin
        n_true = rng.poisson(mean_true)
        if n_true == 0:
            continue
        xt = rng.uniform(logN_lo[b], logN_hi[b], n_true)     # true logN
        Cdet = C_interp(xt, np.full(n_true, snr_cell))
        det = rng.random(n_true) < Cdet                      # completeness thinning
        xt = xt[det]
        xhat = xt + rng.normal(0.0, sigma_kernel, len(xt))   # Eddington smear -> measured
        obs_xhat.append(xhat)
        obs_snr.append(np.full(len(xt), snr_cell))
        obs_zhat.append(rng.uniform(2.4, 2.6, len(xt)))
obs_xhat = np.concatenate(obs_xhat)
obs_snr = np.concatenate(obs_snr)
obs_zhat = np.concatenate(obs_zhat)
n_obs = len(obs_xhat)
assert n_obs > 2000, f"test mis-tuned, only {n_obs} detections"

# the measured-catalog dict the estimator consumes
i_snr = H._cell_index(mm, obs_xhat, obs_snr)[0]
cat_op = dict(xhat=obs_xhat, zhat=obs_zhat,
              sig_x=np.full(n_obs, sigma_kernel),
              sig_z=np.full(n_obs, 1e-4), snr=obs_snr, i_snr=i_snr)

print(f"n_sl (sightlines)   : {n_sl}")
print(f"n_obs (detections)  : {n_obs}   (need > 2000)  -> OK")
print(f"sigma_kernel        : {sigma_kernel} dex  (near-delta, sub-bin)")

n_sl (sightlines)   : 200000
n_obs (detections)  : 4804   (need > 2000)  -> OK
sigma_kernel        : 0.02 dex  (near-delta, sub-bin)


## Assemble $A$, $M$, the penalty, and run the production MAP solve

Now the real estimator pieces, in the same order as the production test:

1. `build_A_ib(...)[1]` → the per-object forward response $A$ (unit-$C$ form +
   metadata; we keep the metadata, index `[1]`).
2. `build_M_b(...)` → the path-length normalizer $M$.
3. `_apply_C_to_A` / `_apply_C_to_M` → fold in the known completeness
   ($\times 1/C$ on $A$, $\times C$ on $M$).
4. Build the **active 2-D index** (which $(N,z)$ cells have support) and the
   $D_2$ smoothing operator with `_build_D2_operator`; map active columns back to
   the flat $(N,z)$ layout.
5. **Warm-start at the injected truth** `x0 = f_b_true` (the solve must *stay*
   near it, not run away) plus a flat fallback start.
6. `_solve_one_lambda(A_act, M_act, lam_fp=0, mu_fp=0, λ=1e-3, D2, [x0, x0_flat])`
   → the MAP heights $f_{\rm best}$. We map them back to the full $N$-grid as
   `f_rec`.

There is **no false-positive background** in this controlled test, so
`lam_fp = 0` and `mu_fp = 0`.

In [6]:
# 1+2: forward response A (keep metadata = index [1]) and normalizer M
A_meta = H.build_A_ib(cat_op, mm, logN_lo, logN_hi, N_b, dN_b, z_edges_fine,
                      Xcalc, cfg, kernel="gaussian")[1]
M_meta = H.build_M_b(qso_zlo, qso_zhi, qso_snr, mm, logN_lo, logN_hi, N_b, dN_b,
                     z_edges_fine, Xcalc, cfg)

# 3: fold in the known completeness  (x1/C on A, xC on M)
A_full = H._apply_C_to_A(A_meta, mm.completeness)
M_full = H._apply_C_to_M(M_meta, mm.completeness)

# 4: active (N,z) cells (occupancy_floor=1 -> any populated column) + D2 penalty
n_zf = len(z_edges_fine) - 1
col_nnz = np.asarray((A_full != 0).sum(axis=0)).ravel().reshape(n_nbins, n_zf)
active_2d = col_nnz > 0
D2, act_idx, n_active = H._build_D2_operator(n_nbins, n_zf, active_2d)

# map each active column back to its flat (jN*n_zf + kz) position
active_flat_cols = np.zeros(n_active, int)
for kz in range(n_zf):
    for jN in range(n_nbins):
        ai = act_idx[jN, kz]
        if ai >= 0:
            active_flat_cols[ai] = jN * n_zf + kz
A_act = A_full[:, active_flat_cols].tocsr()
M_act = M_full[active_flat_cols]

# 5: no FP here; warm-start at the injected truth + a flat fallback start
lam_fp = np.zeros(n_obs)
mu_fp = 0.0
x0 = np.zeros(n_active)
for kz in range(n_zf):
    for jN in range(n_nbins):
        ai = act_idx[jN, kz]
        if ai >= 0:
            x0[ai] = f_b_true[jN]
x0_flat = np.full(n_active, np.median(f_b_true))

# 6: THE production MAP solve
f_best, negP, _ = H._solve_one_lambda(A_act, M_act, lam_fp, mu_fp,
                                      1e-3, D2, [x0, x0_flat])

# map the active solution back onto the full N-grid
f_rec = np.zeros(n_nbins)
for kz in range(n_zf):
    for jN in range(n_nbins):
        ai = act_idx[jN, kz]
        if ai >= 0:
            f_rec[jN] += f_best[ai]

print(f"n_active cells      : {n_active}")
print(f"negP (MAP -logpost) : {negP:.4g}")
print(f"f_rec (head)        : {np.array2string(f_rec[:4], precision=3)}")

n_active cells      : 14
negP (MAP -logpost) : 1.557e+04
f_rec (head)        : [2.822e-22 1.841e-22 1.234e-22 8.622e-23]


## Visualize: injected vs recovered

Left: the recovered CDDF heights $f_{\rm rec}$ over the injected truth
$f_b^{\rm true}$ (log-$y$). Right: the integrated $dN/dX(\ge 20.3)$ — truth vs
recovery as two bars. If the deconvolution is doing its job, the curves overlay
and the bars match.

In [7]:
logN_ctr = 0.5 * (logN_lo + logN_hi)
sel = logN_lo >= 20.3 - 1e-9
dndx_rec = float(np.sum(f_rec[sel] * dN_b[sel]))
dndx_true = float(np.sum(f_b_true[sel] * dN_b[sel]))

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.2))

axL.plot(logN_ctr, f_b_true, "k-", lw=2, label="injected truth $f_b^{true}$")
axL.plot(logN_ctr, f_rec, "o", ms=6, color="tab:red", label="recovered $f_{rec}$")
axL.axvline(20.3, ls=":", color="gray", label="20.3 report floor")
axL.set_yscale("log")
axL.set_xlabel(r"$\log_{10} N_{\rm HI}$")
axL.set_ylabel(r"$f(N)$")
axL.set_title("CDDF heights: injected vs MAP-recovered")
axL.legend()

axR.bar([0, 1], [dndx_true, dndx_rec], color=["k", "tab:red"], width=0.6)
axR.set_xticks([0, 1])
axR.set_xticklabels(["truth", "recovered"])
axR.set_ylabel(r"$dN/dX\,(\geq 20.3)$")
axR.set_title(f"integrated $dN/dX(\\geq 20.3)$\nratio = {dndx_rec/dndx_true:.3f}")

fig.tight_layout()
plt.show()

/tmp/ipykernel_2908878/1842260186.py:24: UserWarning: FigureCanvasPdf is non-interactive, and thus cannot be shown
  plt.show()


## The closure self-check

The same assertion the production test makes: the deconvolved integrated
$dN/dX(\ge 20.3)$ must match the **injected truth** within 8%.

In [8]:
sel = logN_lo >= 20.3 - 1e-9
dndx_rec = float(np.sum(f_rec[sel] * dN_b[sel]))
dndx_true = float(np.sum(f_b_true[sel] * dN_b[sel]))
ratio = dndx_rec / dndx_true

assert abs(dndx_rec - dndx_true) / dndx_true < 0.08, (
    f"closure off: rec={dndx_rec:.5g} true={dndx_true:.5g} (ratio {ratio:.3f})")

print(f"dN/dX(>=20.3) true  : {dndx_true:.6g}")
print(f"dN/dX(>=20.3) rec   : {dndx_rec:.6g}")
print(f"ratio rec/true      : {ratio:.4f}")
print(f"n_obs               : {n_obs}")
print(f"|ratio - 1|         : {abs(ratio - 1):.4f}  (<0.08 required)")
print("\nCLOSURE PASS — the MAP deconvolution recovered the injected truth.")

dN/dX(>=20.3) true  : 0.0216702
dN/dX(>=20.3) rec   : 0.0215262
ratio rec/true      : 0.9934
n_obs               : 4804
|ratio - 1|         : 0.0066  (<0.08 required)

CLOSURE PASS — the MAP deconvolution recovered the injected truth.


## Honesty: what this test does and does not prove

Read this before you over-interpret the green checkmark.

**(a) Why the self-check is non-circular here — and where circularity *would*
creep in.** The assertion above compares the recovered $dN/dX$ to the
**injected** truth — a quantity we chose, completely independent of the catalog
the estimator saw. That is a genuine closure test of the forward model + solve.

The thing to *not* do is confuse this with an "$\alpha = 1/R_0$" recovery. On a
*real mock*, one fits the completeness/kernel on the same truth used to *score*
the recovery; then the recovered-over-truth ratio is driven to 1 essentially **by
construction** — a tautology, not evidence. The honest external test of the
method is **cross-mock transfer**: build the kernel on one mock, apply it to a
held-out mock of a *different* recipe, and check $R_0(z)\approx 1$ *without*
refitting. That is **NB4** (future) — not this notebook.

**(b) There is no error band here, and the band you'll see elsewhere is
statistical only.** This notebook reports a single point recovery. The
Monte-Carlo band shown in **NB3** quantifies *statistical* uncertainty only — it
is **not** the total systematic $\sigma_{\rm tot}$. Completeness/kernel
mis-specification and the Eddington high-$N$ amplification are separate
systematics that the band does not capture.

**(c) `resp_kind="kappa"` is a legacy default — do not audit the dataclass
default and call it the method.** `HBIConfig` defaults `resp_kind="kappa"`, which
builds $A$ from the cached **GP-posterior** $\kappa$ object purely for
byte-identical backward reproduction. The **headline science pipeline sets
`resp_kind="forward"`**, which builds $A$ from the *forward likelihood*
$p(\hat N\mid N,{\rm SNR},z)$ — the object that removes the narrow-kappa high-$N$
over-recovery. This synthetic notebook supplies its *own* Gaussian forward kernel
directly via `build_A_ib(..., kernel="gaussian")`, so it does not exercise either
production response path; it exercises the **normalization and solve mechanics**.
Auditing the dataclass default in isolation mis-reconstructs the production
method — read NB2 for the real forward kernel.

## Recap and where to go next

We injected a known power-law $f(N)$, forward-simulated a completeness-thinned,
Eddington-smeared detection catalog, fed it to the **real production MAP
deconvolution** (`cddf_catalog_hbi._solve_one_lambda` with the production
$A$/$M$/penalty assembly), and recovered the injected $dN/dX(\ge 20.3)$ within
8% — all in a couple of seconds, with no GP inference, no caches, no GPFS.

The near-delta ($\sigma=0.02$ dex) kernel made this the $\sigma\to 0$ anchor,
where the deconvolution reduces to a $1/V_{\rm max}$ count and the test isolates
the forward-model normalization and solve mechanics.

**Next:**
- **NB2** — replace the synthetic Gaussian forward kernel with the *real* 2LPT-0
  forward kernel (`resp_kind="forward"`) and watch the genuine finite-width
  deconvolution at work.
- **NB3** — open up the marked-Poisson likelihood, the MAP solve internals, and
  the Monte-Carlo statistical band.